This notebook is used to calculate the direction, spacing, interaction index, slopes, y-junction bisectors and other related parameters of dunes within a 0.1 by 0.1 degree cell (equivalent to the ERA5-Land grid).

# Imports

In [ ]:
import glob
import pickle
import geopandas as gpd
import numpy as np 
import pandas as pd
from shapely import Polygon, LineString, MultiLineString, Point, GeometryCollection, MultiPoint, line_merge, snap
from shapely.ops import split
from shapely.ops import unary_union
from shapely import get_point, overlaps, line_merge
from affine import Affine
import rasterio
from rasterio.windows import Window,from_bounds
import matplotlib.pyplot as plt
from scipy.stats import circmean, circstd
from scipy.interpolate import RegularGridInterpolator,  interp1d 
from scipy.signal import find_peaks

from YjunctionClass import Yjunction

import warnings
warnings.filterwarnings("error")
warnings.filterwarnings("ignore", message="Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.") # Silence this shapefile warning

# Functions

Functions for calculations. A few of them are redundant now and should be cleaned in the future.

In [ ]:
def collection_to_multilinestring(clipped_geom):
    if not isinstance(clipped_geom,GeometryCollection):
        return clipped_geom
    return MultiLineString([geom for geom in clipped_geom.geoms if (isinstance(geom,LineString) or isinstance(geom,MultiLineString))])

def limit_neg_pi_less_0(ang):
    if ang>=0:
        ang = ang - np.pi
    if ang==0:
        ang = -np.pi
    return ang

def range_neg_pi_to_pi(ang):
    "Translate 2 pi to -pi to pi"
    return (ang + np.pi) % (2*np.pi) - np.pi
    
def limit_neg_pi_less_0_array(angs):
    # should result in all values being between -np.pi and smaLler 0
    angs[angs>=0] = angs[angs>=0]-np.pi
    angs[angs==0] = -np.pi
    
    return angs

def generate_transects(row, trans_width=0.1, Nt=25):
    ang = row.linear_ang
    lon = row.lon
    lat = row.lat

    # create Nt evenly spaced points parallel to the average crest orientation
    tlon = np.linspace(lon-np.cos(ang)*trans_width*2**-0.5,lon+np.cos(ang)*trans_width*2**-0.5,Nt)
    tlat = np.linspace(lat-np.sin(ang)*trans_width*2**-0.5,lat+np.sin(ang)*trans_width*2**-0.5,Nt)

    # create perpendicular transects to the average dune orientation based of the above
    tlon_rot_min = tlon-np.cos(ang+np.pi/2)*trans_width*2**-0.5
    tlat_rot_min = tlat-np.sin(ang+np.pi/2)*trans_width*2**-0.5
    tlon_rot_max = tlon+np.cos(ang+np.pi/2)*trans_width*2**-0.5
    tlat_rot_max = tlat+np.sin(ang+np.pi/2)*trans_width*2**-0.5

    # generate a GeoDataFrame of these lines and populate with LineStrings
    df_transect = gpd.GeoDataFrame(np.stack([tlon_rot_min, tlat_rot_min, tlon_rot_max, tlat_rot_max], axis=1),
                               columns=['trans_x_min','trans_y_min','trans_x_max','trans_y_max'])
    df_transect["transects"] = df_transect.apply(lambda x: LineString([[x.trans_x_min,x.trans_y_min],[x.trans_x_max,x.trans_y_max]]),axis=1)
        
    trans_multi = MultiLineString(df_transect.transects.to_list())
    
    return trans_multi

def get_junctions(lines):

    # Duplicate the geometry to allow spatial join [geom_left,geom_right]
    lines['geom'] = lines.geometry
    gdf = gpd.sjoin(lines, lines,
                    how="inner",
                    predicate="intersects",
                    lsuffix="left",
                    rsuffix="right")

    # Remove geometries that intersect themselves
    gdf = gdf[gdf.index != gdf.index_right]

    # Get the intersection points and set them as geometry 
    gdf['intersection_geom'] = gdf['geom_left'].intersection(gdf['geom_right'])
    
    #inserted this because of future warning
    gdf_old_geo_name = gdf.active_geometry_name
    gdf = gdf.set_geometry('intersection_geom', crs=lines.crs).drop(gdf_old_geo_name,axis=1).rename_geometry(gdf_old_geo_name)

    # this is the old version of the above
    # gdf.set_geometry('intersection_geom', drop=True, inplace=True, crs=lines.crs)
    gdf = gdf.drop_duplicates(subset=['geometry'])["geometry"]

    gdf = gpd.GeoDataFrame(geometry=list(gdf.union_all().geoms)).set_crs(lines.crs)
    gdf["geom"] = gdf.geometry

    # perform a spatial join between all lines and the unctions points to get unctions as MultiLines
    gdf = gpd.sjoin(lines, gdf,
                    how="inner",
                    predicate="intersects",
                    lsuffix="left",
                    rsuffix="right")

    gdf = gdf.dissolve("index_right").drop("geom_left",axis=1).rename(columns={"geom_right":"J_points"})
    gdf.index.names = ['Index']

    starts = get_point(lines.geometry,0).to_frame().set_crs(lines.crs)
    ends = get_point(lines.geometry,-1).to_frame().set_crs(lines.crs)
    start_ends = pd.concat([starts,ends])
    jpnts = gdf.set_geometry(gdf.J_points).geometry.to_frame().set_crs(gdf.crs)
    gdf_terms = start_ends[~start_ends.geometry.isin(jpnts.geometry)]

    # explode all multilines to find the multi junctions
    gdf_junctions_exp = gdf.explode(index_parts=False)
    first_level = gdf_junctions_exp.index.get_level_values(0)
    counts = first_level.value_counts()
    filtered = counts[counts >= 4]
    result = filtered.index.tolist()
    
    # split multi junctions and y junctions these have also the lines to calculate y opening!!
    gdf_multi = gdf[gdf.index.get_level_values(0).isin(result)]
    gdf_yj = gdf[~gdf.index.get_level_values(0).isin(result)]
    
    return gdf, gdf_yj, gdf_multi, gdf_terms

def length_geom(geom):
    try:
        no_geoms = len(geom.geoms)
    except AttributeError:
        no_geoms = 0
    return no_geoms

def save_tif(img, transform, crs, filename, nodata=np.nan):
    """save an image using rasterio"""
    height, width = img.shape
    with rasterio.open(
        filename,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=img.dtype,
        crs=crs,
        transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(img, 1)
        
def generate_linestring(lon,lat,ang,width=0.1*2**-0.5):
    if np.isnan(ang):
        return 
    tlonmin, tlonmax = lon-np.cos(ang)*width, lon+np.cos(ang)*width
    tlatmin, tlatmax = lat-np.sin(ang)*width, lat+np.sin(ang)*width
    line = LineString([[tlonmin,tlatmin],[tlonmax,tlatmax]])

    return line
    
def andrew_ang_line(line):
    linel=line.length

    xs = np.asarray(line.xy[0])
    ys = np.asarray(line.xy[1])

    if len(np.unique(xs))>1:
        if len(xs)>2:
            linea0 = np.arctan(np.polyfit(xs,ys,deg=1)[0])
        else:
            linea0 = np.arctan((ys[1]-ys[0])/(xs[1]-xs[0]))
    else:
        linea0 = np.pi/2

    if len(np.unique(ys))>1:
        if len(ys)>2:
            linea1 = np.arctan(np.polyfit(ys,-xs,deg=1)[0])
        else:
            linea1 = np.arctan((xs[0]-xs[1])/(ys[1]-ys[0]))
    else:
        linea1 = np.pi/2

    return linea0,linea1,linel

def andrew_ang_group(linea0,linea1,linel):
    if circstd(linea1)>circstd(linea0):
        crestangleavgs = np.angle(np.sum(np.exp(1j*linea0)*linel))
        crestanglestds = circstd(linea0)
    else:
        crestangleavgs = np.angle(np.sum(np.exp(1j*(linea1+np.pi/2))*linel))
        crestanglestds = circstd(linea1)
    
    return crestangleavgs, crestanglestds

def check_junction_presence(gdf, key):
    try:
        return gdf.loc[key]
    except KeyError:
        return None

def get_interaction_points(row, crs, junctions):
    
    gdf = gpd.sjoin(row, row, how='inner',predicate="intersects",lsuffix="left",rsuffix="right")
    gdf = gdf[gdf.index != gdf.index_right]
    gdf["intersection_geom"] = gdf['geom_left'].intersection(gdf['geom_right'])
    gdf_old_geo_name = gdf.active_geometry_name
    gdf = gdf.set_geometry('intersection_geom', crs=row.crs).drop(gdf_old_geo_name,axis=1).rename_geometry(gdf_old_geo_name)
    gdf =  gdf.drop_duplicates(subset=['geometry'])["geometry"]

    if gdf.empty:
        return
    
    try:
        gdf = gpd.GeoDataFrame(geometry=list(gdf.union_all().geoms),crs=row.crs)
    except AttributeError:
        # handles the one polygon case of a union
        gdf = gpd.GeoDataFrame(geometry=[gdf.union_all()],crs=row.crs)

    gdf["interaction_point"] = gdf.centroid.to_crs(crs)
    gdf.to_crs(crs, inplace=True)

    #exceutes when there are no junctions to compare
    if junctions is None:
        gdf_old_geo_name = gdf.active_geometry_name
        gdf = gdf[gdf.interaction_point.intersects(row.geom_poly.iloc[0])].set_geometry("interaction_point",drop=True).drop(gdf_old_geo_name,axis=1)#.rename_geometry("interaction_point")
        return 

    intersect_poly_junctions_join = gpd.sjoin(gdf,junctions)
    junctions_idx = intersect_poly_junctions_join.index
    not_junctions_idx = gdf.index.difference(junctions_idx)
    gdf_old_geo_name = gdf.active_geometry_name
    gdf = gdf.loc[not_junctions_idx].set_geometry("interaction_point").drop(gdf_old_geo_name,axis=1)#.rename_geometry("interaction_point")
    junctions_interact_points = junctions.loc[intersect_poly_junctions_join.index_right.values].rename_geometry("interaction_point")
    gdf = pd.concat([gdf,junctions_interact_points])
    gdf = gdf[gdf.interaction_point.intersects(row.geom_poly.iloc[0])]

    print("\r%05d"%(row.iloc[0,0]), end='')
    
    return gdf

def update_empty_interaction(multi,yj):
    if not multi is None:
        multi_pnts = list(multi.geoms)
    else:
        multi_pnts = []

    if not yj is None:
        yj_pnts = list(yj.geoms)
    else:
        yj_pnts = []
    
    return MultiPoint(multi_pnts+yj_pnts)

def gdf_var_to_raster(gdf, var, filename, nodata, lon, lat, crs=4326):
    gdf_pivot = gdf.pivot(index="lat", columns="lon", values=var)
    
    # Determine the missing indices and columns
    missing_indices = set(lat) - set(gdf_pivot.index)
    missing_columns = set(lon) - set(gdf_pivot.columns)
    
    # Expand the DataFrame
    gdf_pivot = gdf_pivot.reindex(index=list(gdf_pivot.index) + list(missing_indices), 
                    columns=list(gdf_pivot.columns) + list(missing_columns))
    gdf_pivot = gdf_pivot.sort_index(axis=0).sort_index(axis=1)

    # Define raster parameters
    x_min, y_max = gdf_pivot.columns[0], gdf_pivot.index[-1]  # Upper-left corner (x_min, y_max)
    x_res, y_res = 0.1, 0.1    # Resolution (pixel width and height)
    
    # Create the affine transform but shift by half so raster lies on top in qgis
    transform = Affine.translation(x_min-0.5*x_res, y_max+0.5*y_res) * Affine.scale(x_res, -y_res)
    
    save_tif(np.flipud(gdf_pivot),transform,crs,filename,nodata=nodata)

def get_bisectors_for_line(pnt,line_series,radius=[0.51]):
    if len(line_series)<3:
        return np.ones(10)*np.nan
    yj = Yjunction(pnt,line_series)
    yj.get_radius_intersects(radius)
    yj.get_angle_intersects()
    yj.get_angle_between_points()
    # print(yj.get_angle_between_points())
    yj.get_bisectors_between_points()
    junction_params = yj.get_junction_opening_and_direction()
    oangs_bdirs=[]
    for k0,v0 in junction_params.items():
        for k1,v1 in v0.items():
            oangs_bdirs.append(float(v1["opening_angle"]))
            oangs_bdirs.append(float(v1["bisector_direction"]))
    if len(oangs_bdirs)<10:
        print("problem?")
    return oangs_bdirs

def sign_crests(crest, bisector):
    # cases where crest greater than bisector
    mask1 = (crest > bisector) & ((crest - bisector) <= np.pi / 2)
    mask2 = (crest > bisector) & ((crest - bisector) > 1.5 * np.pi)

    # cases where bisector greater than crest
    mask3 = (bisector > crest) & ((bisector - crest) <= np.pi / 2)
    mask4 = (bisector > crest) & ((bisector - crest) > 1.5 * np.pi)

    # combine masks where the transformation is the same
    flip_mask = (mask1 | mask2 | mask3 | mask4 | (crest == bisector))

    # apply transformation
    new_crest = np.where(flip_mask, range_neg_pi_to_pi(crest + np.pi), crest)

    return new_crest

### FOR THE SLOPES ###

def segment_extract_mean(src, line):
    img, _ = rasterio.mask.mask(src, [line], all_touched=True, crop=True, nodata=np.nan)
    return np.nanmean(img)

def segment_extract_max(src, line):
    img, _ = rasterio.mask.mask(src, [line], all_touched=True, crop=True, nodata=np.nan)
    return np.nanmax(img)

def get_transect(pnt,ang,xy_width=90,segs=3):
    lon,lat = pnt.xy
    points = np.linspace(xy_width/2,-xy_width/2,segs)
    x = lon-np.cos(ang) * points
    y = lat-np.sin(ang) * points
    line = LineString(np.stack((x,y),axis=1))
    return line

def get_cop_files(lon, lat, cop_dict, pad=0.1):
    left = 0
    right = 0
    for key, value in cop_dict.items():
        east=float(value[-18:-12])
        west=float(value[-10:-4])
        if (east <= (lon-pad)):
            left=key
        if (west >= (lon+pad)):
            right=key
            if left == right:
                return [left]
            return [left,right]

def ang_new(x_left,x_right,y_left,y_right):
    xs = np.array([x_left,x_right])
    ys = np.array([y_left,y_right])
    
    if not len(np.unique(xs))<2:
        if not len(np.unique(ys))<2:
            ang = limit_neg_pi_less_0(np.arctan((ys[1]-ys[0])/(xs[1]-xs[0])))
        else:
            ang=-np.pi
    else:
        ang = -np.pi/2
        
    return ang

def get_segments(line):
    x_left = np.repeat(line.coords.xy[0],2)[1:-1].reshape(-1,2)[:,0]
    x_right = np.repeat(line.coords.xy[0],2)[1:-1].reshape(-1,2)[:,1]
    y_left = np.repeat(line.coords.xy[1],2)[1:-1].reshape(-1,2)[:,0]
    y_right = np.repeat(line.coords.xy[1],2)[1:-1].reshape(-1,2)[:,1]
    data = {"x_left":x_left,"x_right":x_right,"y_left":y_left,"y_right":y_right}
    df = pd.DataFrame(data)
    
    return df

def cops_idx_to_filename(cop_idx, cop_dict):
    cop_filenames = []
    for i in cop_idx:
        cop_filenames.append(cop_dict[i])
    return cop_filenames
    
def era5_pad_window(cop_files, lon, lat, pad=0.05):
    minx, maxx = lon-pad, lon+pad
    miny, maxy = lat-pad, lat+pad

    if len(cop_files)==2:
        with rasterio.open(cop_files[0]) as src1, rasterio.open(cop_files[1]) as src2:

            # SRC1
            wnd_row_off_min1, wnd_col_off_min1 = src1.index(minx, maxy)
            wnd_row_off_max1, wnd_col_off_max1 = src1.index(maxx, miny)
            wnd_col_off_max1 = src1.width
            window1 = Window(wnd_col_off_min1,wnd_row_off_min1,wnd_col_off_max1 - wnd_col_off_min1,wnd_row_off_max1 - wnd_row_off_min1)
            
            transform1 = src1.window_transform(window1)
            img1 = src1.read(1,window=window1)
        
            meta1 = src1.meta.copy()
            meta1.update({"height": window1.height,"width": window1.width,"transform": transform1})
            
            rows_mask1 = np.arange(0, img1.shape[0])
            cols_mask1 = np.arange(0, img1.shape[1])

            _, lats_mask1 = rasterio.transform.xy(transform1, rows_mask1, [0])
            lons_mask1, _ = rasterio.transform.xy(transform1, [0], cols_mask1)
        
            # SRC2
            wnd_row_off_min2, wnd_col_off_min2 = src2.index(minx, maxy)
            wnd_row_off_max2, wnd_col_off_max2 = src2.index(maxx, miny)
            wnd_col_off_min2 = 0
            window2 = Window(wnd_col_off_min2,wnd_row_off_min2,wnd_col_off_max2 - wnd_col_off_min2,wnd_row_off_max2 - wnd_row_off_min2)
            
            transform2 = src2.window_transform(window2)
            img2 = src2.read(1,window=window2)
        
            meta2 = src2.meta.copy()
            meta2.update({"height": window2.height,"width": window2.width,"transform": transform2})
        
            rows_mask2 = np.arange(0, img2.shape[0])
            cols_mask2 = np.arange(0, img2.shape[1])

            _, lats_mask2 = rasterio.transform.xy(transform2, rows_mask2, [0])
            lons_mask2, _ = rasterio.transform.xy(transform2, [0], cols_mask2)

            lons_mask = np.concatenate((lons_mask1,lons_mask2))
            lats_mask = np.array(lats_mask1) #could be any
            img = np.concatenate((img1,img2),axis=1)
            
    else:
        with rasterio.open(cop_files[0]) as src:

            wnd_row_off_min, wnd_col_off_min = src.index(minx, maxy)
            wnd_row_off_max, wnd_col_off_max = src.index(maxx, miny)
            window = Window(wnd_col_off_min,wnd_row_off_min,wnd_col_off_max - wnd_col_off_min,wnd_row_off_max - wnd_row_off_min)
            
            transform = src.window_transform(window)
            img = src.read(1,window=window)

            meta = src.meta.copy()
            meta.update({"height": window.height,"width": window.width,"transform": transform})

            rows_mask = np.arange(0, img.shape[0])
            cols_mask = np.arange(0, img.shape[1])

            _, lats_mask = np.array(rasterio.transform.xy(transform, rows_mask, [0]))
            lons_mask, _ = np.array(rasterio.transform.xy(transform, [0], cols_mask))
            # lons_mask_epsg3577, lats_mask_epsg3577 = rasterio.warp.transform(src.crs,"EPSG:3577",lons_mask,lats_mask)

    # return lons_mask_epsg3577,lats_mask_epsg3577,img,meta
    return lons_mask,lats_mask,img

def get_centroid(x_left,x_right,y_left,y_right):
    return Point(x_left-(x_left-x_right)/2,y_left-(y_left-y_right)/2)

def get_peak_troughs(trans_heights, index, centroid_idx):
    
    # centroid_idx = len(trans_heights)//2
    peaks_idx = find_peaks(trans_heights)[0]
    troughs_idx = find_peaks(-trans_heights)[0]
    if len(peaks_idx) == 0:
        return -1, -1, -1
    try:
        centroid_is_peak = ((trans_heights[centroid_idx-1] < trans_heights[centroid_idx]) and (trans_heights[centroid_idx] > trans_heights[centroid_idx+1]))
    except IndexError:
        return -1, -1, -1
        
    higher_idx_is_upslope = trans_heights[centroid_idx+1] > trans_heights[centroid_idx]
    lower_idx_is_upslope = trans_heights[centroid_idx-1] > trans_heights[centroid_idx]
    
    # if trough for some reason
    if (higher_idx_is_upslope and lower_idx_is_upslope):
        return -1, -1, -1

    try:
        if centroid_is_peak:
            peak = centroid_idx 
        elif higher_idx_is_upslope:
            peak = peaks_idx[peaks_idx>centroid_idx][0]
        else:
            peak = peaks_idx[peaks_idx<centroid_idx][-1]
    except IndexError:
        return -1, -1, -1

    try:
        lower_trough = troughs_idx[troughs_idx<peak][-1]
    except IndexError:
        lower_trough = 0
    
    try:
        upper_trough = troughs_idx[troughs_idx>peak][0]
    except IndexError:
        upper_trough = len(trans_heights)-1

    return lower_trough, peak, upper_trough

def isolate_dune(trans_ocrs, trans_heights, lower_trough_idx, peak_idx, upper_trough_idx, threshold):
    if np.isnan(threshold):
        return np.nan, np.nan, np.nan, np.nan
    x_trans = np.linspace(0,trans_ocrs.length,len(trans_heights))
    x_trans = x_trans - x_trans[peak_idx]

    mask = trans_heights[lower_trough_idx:upper_trough_idx+1]>=(threshold)
    isolated_height = trans_heights[lower_trough_idx:upper_trough_idx+1][mask]
    isolated_x = x_trans[lower_trough_idx:upper_trough_idx+1][mask]

    new_peak_idx = np.argwhere(isolated_height == trans_heights[peak_idx])[0,0]

    upper_min = isolated_height[new_peak_idx:].min()
    lower_min = isolated_height[:new_peak_idx+1].min()
    
    if upper_min>=lower_min:
        isolated_x = isolated_x[isolated_height>=upper_min]
        isolated_height = isolated_height[isolated_height>=upper_min]
    else:
        isolated_x = isolated_x[isolated_height>=lower_min]
        isolated_height = isolated_height[isolated_height>=lower_min]
    
    final_peak_idx = np.argwhere(isolated_height == trans_heights[peak_idx])[0,0]
    
    return x_trans, isolated_x, isolated_height, final_peak_idx

def relative_height_thresh(row):
    peak = row.trans_height[row.peak_idx]
    lower_trough = row.trans_height[row.lower_trough_idx]
    upper_trough = row.trans_height[row.upper_trough_idx]

    if (np.isnan(lower_trough) and np.isnan(upper_trough)):
        threshold = np.nan
    elif np.isnan(lower_trough):
        threshold = (peak-upper_trough)/2
    elif np.isnan(upper_trough):
        threshold = (peak-lower_trough)/2
    else:
        if lower_trough < upper_trough:
            threshold = (peak-upper_trough)/2
        else:
            threshold = (peak-lower_trough)/2

    threshold = peak-threshold
    return threshold

def find_nearest_crest_points(row, cx_lines, cell_ang):

    bounds = cx_lines.bounds

    lon_min = bounds[0]
    lon_max = bounds[2]
    lat_min = bounds[1]
    lat_max = bounds[3]
    lat_width = abs(lat_max-lat_min)
    lon_width = abs(lon_max-lon_min)
    if (lat_width > lon_width):
        width = lat_width/2
    else:
        width = lon_width/2

    # since -pi =< ang < 0 adding half pi ensures to get transect left of the "orientation"
    ang = row.angs_3577+(np.pi/2)

    # check what would be left and right of the crest:
    angdiff = abs(cell_ang - row.angs_3577)
    if ((angdiff>(np.pi/2)) and (angdiff<(1.5*np.pi))):
        flippnts = True
    else:
        flippnts = False
    
    lon,lat = row.centroids.x,row.centroids.y

    #line1
    x1 = lon-np.cos(ang) * width
    y1 = lat-np.sin(ang) * width
    x1 = np.array([x1,lon])
    y1 = np.array([y1,lat])
    line1 = LineString(np.stack((x1,y1),axis=1))

    multipnt1 = cx_lines.intersection(line1)

    if multipnt1.is_empty:
        return Point()
        
    if isinstance(multipnt1,LineString):
        return Point()

    if isinstance(multipnt1,MultiLineString):
        return Point()
        
    if isinstance(multipnt1,GeometryCollection):
        multipnt1 = gpd.GeoSeries(multipnt1.geoms).explode()
        multipnt1 = multipnt1[multipnt1.geom_type=="Point"].values
        if len(multipnt1) == 0:
            return Point()
        elif len(multipnt1) == 1:
            multipnt1 = multipnt1[0]
        else:
            multipnt1 = MultiPoint(multipnt1)
        
    if isinstance(multipnt1, MultiPoint):
        multipnt1 = gpd.GeoSeries(list(multipnt1.geoms))
        multipnt1_dist = multipnt1.distance(row.centroids).sort_values()
        try:
            pnt1 = multipnt1.loc[multipnt1_dist[multipnt1_dist > 1].index[0]]
        except IndexError:
            return Point()
    elif isinstance(multipnt1, Point):
        # the single point that got identified is actually the centroid return empty
        if (abs(row.centroids.distance(multipnt1)) < 1):
            return Point()
        pnt1 = multipnt1
    
    #line2
    x2 = lon+np.cos(ang) * width
    y2 = lat+np.sin(ang) * width
    x2 = np.array([lon,x2])
    y2 = np.array([lat,y2])
    line2 = LineString(np.stack((x2,y2),axis=1))

    multipnt2 = cx_lines.intersection(line2)

    if multipnt2.is_empty:
        return Point()

    if isinstance(multipnt2,LineString):
        return Point()

    if isinstance(multipnt2,MultiLineString):
        return Point()
        
    if isinstance(multipnt2,GeometryCollection):
        multipnt2 = gpd.GeoSeries(multipnt2.geoms).explode()
        multipnt2 = multipnt2[multipnt2.geom_type=="Point"].values
        if len(multipnt2) == 0:
            return Point()
        elif len(multipnt2) == 1:
            multipnt2 = multipnt2[0]
        else:
            multipnt2 = MultiPoint(multipnt2)
            
    if isinstance(multipnt2, MultiPoint):
        multipnt2 = gpd.GeoSeries(list(multipnt2.geoms))
        multipnt2_dist = multipnt2.distance(row.centroids).sort_values()
        try:
            pnt2 = multipnt2.loc[multipnt2_dist[multipnt2_dist > 1].index[0]]
        except IndexError:
            return Point()
    elif isinstance(multipnt2, Point):
        # the single point that got identified is actually the centroid return empty
        if (abs(row.centroids.distance(multipnt2)) < 1):
            return Point()
        pnt2 = multipnt2

    if not flippnts:
        return MultiPoint([pnt2, pnt1])
    else:
        return MultiPoint([pnt1, pnt2])

def find_centroid_idx(line, pnt):
    "find the index of the centroid across the line"
    return np.argmin(abs(np.sum(np.array(list(line.coords))-(pnt.x, pnt.y),axis=1)))

def get_ang_from_pnts(ppnt, npnt):
    x_left = ppnt.x
    x_right = npnt.x
    y_left = ppnt.y
    y_right = npnt.y
    ang = float(ang_new(x_left,x_right,y_left,y_right))
    return ang

def samp_line(line, step=30, offset=15, poff=.001, noff=.001):
    spnts = line.interpolate(np.arange(offset,line.length-offset,step))

    if ((len(spnts)==0) or (len(spnts)==1)):
        spnts = line.interpolate([line.length/2])
        ppnts = line.interpolate([line.length/2-poff])
        npnts = line.interpolate([line.length/2+noff])
    else:
        ppnts = line.interpolate(np.arange(offset-poff,line.length-offset,step))
        npnts = line.interpolate(np.arange(offset+noff,line.length-offset,step))
        
        if not (len(ppnts) == len(spnts) == len(npnts)):
            if (len(ppnts) > len(npnts)):
                ppnts = ppnts[:-abs(len(ppnts) - len(npnts))]
            elif (len(ppnts) < len(npnts)): #should never happen
                npnts = npnts[:-abs(len(ppnts) - len(npnts))]
    
            # can just check ppnts (same as npnts in len)
            if (len(ppnts) > len(spnts)):
                ppnts = ppnts[:-abs(len(ppnts) - len(spnts))]
                npnts = npnts[:-abs(len(npnts) - len(spnts))]
            elif (len(ppnts) < len(spnts)):
                spnts = spnts[:-abs(len(ppnts) - len(spnts))]
            
            # if above results in less than two sample points (like above)
            if (len(spnts)<2):
                spnts = line.interpolate([line.length/2])
                ppnts = line.interpolate([line.length/2-poff])
                npnts = line.interpolate([line.length/2+noff])

    df = pd.DataFrame({"ppnts":ppnts,"npnts":npnts})
    df["angs_3577"] = df.apply(lambda x: get_ang_from_pnts(x.ppnts,x.npnts), axis=1)
    df["centroids"] = spnts
    return df.drop(["ppnts","npnts"],axis=1)

def get_isolated_area(x, height, peakidx):
    """Return twice the area from lowest throught to peak"""
    height -= height.min() #relative height
    if (height[0] <= height[-1]):
        area = np.trapezoid(height[0:peakidx+1],x[0:peakidx+1])
    else:
        area = np.trapezoid(height[peakidx:],x[peakidx:])
    return float(2*area)

def handle_rasterio(row, cop_dict, pad=0.055):
    print("\r%05d"%(row.name), end="")
    cop_files = cops_idx_to_filename(row["COPs"],cop_dict)
    lons,lats,img = era5_pad_window(cop_files,row.lon,row.lat,pad)
    inter = RegularGridInterpolator((lats,lons), img, bounds_error=False, fill_value=np.nan, method="cubic") # lats first!
    
    try:
        gdf_lines = gpd.GeoDataFrame(geometry=list(row.clipped_lines_3577.geoms),crs=3577) 
    except AttributeError:
        gdf_lines = gpd.GeoDataFrame(geometry= [MultiLineString([row.clipped_lines_3577])],crs=3577)
    gdf_lines = gdf_lines[~((gdf_lines.geom_type=="Point") | (gdf_lines.geom_type=="MultiPoint"))]
    df = pd.concat(gdf_lines.geometry.apply(lambda x: samp_line(x,500,50)).to_list(), axis=0) # determines the spacing and offset of sampling THIS CAN BE altered!!!
    df["group"] = (df.index == 0).cumsum() - 1
    index_dict = dict(enumerate(list(gdf_lines.index),0))
    df['index'] = df.group.map(index_dict)
    df = df.set_index(['index', df.groupby('index').cumcount()])
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in intersection")
        df["neighbours"] = df.apply(lambda x: find_nearest_crest_points(x, row.geometry_3577, row.linear_ang),axis=1) #potential timesafe
    # THIS PART HERE MIGHT NEED SO FURTHWER THINKING TO CHEKC THAT STEPS ARE NOT IGNORED BECAUS OF MISSING DUNE CRESTS ON EITHER SIDE MATYBE ONE SIDE IS GOODF?
    df = df[~gpd.GeoSeries(df.neighbours).is_empty]
    if df.empty:
        return row.linear_ang, np.nan, np.nan
    df["trans_3577"] = df.apply(lambda x: LineString([[x.neighbours.geoms[0].x,x.neighbours.geoms[0].y],
                                                      [x.centroids.x, x.centroids.y],
                                                      [x.neighbours.geoms[1].x,x.neighbours.geoms[1].y]]).segmentize(1),axis=1) # potential timesafe
    df["trans_4326"] = gpd.GeoSeries(df.trans_3577).set_crs(3577).to_crs(4326)
    df["centroid_idx_3577"] = df.apply(lambda x: find_centroid_idx(x.trans_3577,x.centroids), axis=1)
    df["trans_height"] = df.trans_4326.apply(lambda x: inter((x.xy[1],x.xy[0])))
    dunes_idxs = df.apply(lambda x: get_peak_troughs(x.trans_height, x.name, x.centroid_idx_3577),axis=1)
    df[["lower_trough_idx","peak_idx","upper_trough_idx"]] = pd.DataFrame(dunes_idxs.to_list(),index=df.index)
    df = df[~(df.peak_idx==-1)] # get rid of transects that have no peaks
    if df.empty:
        return row.linear_ang, np.nan, np.nan
    isolated_dune = df.apply(lambda x: isolate_dune(x.trans_3577, x.trans_height,
                                                      x.lower_trough_idx, x.peak_idx, 
                                                      x.upper_trough_idx, relative_height_thresh(x)), axis=1)
    df[["x_trans_height", "isolated_x", "isolated_height", "isolated_peak_idx"]] = pd.DataFrame(isolated_dune.to_list(),index=df.index)
    df = df[~(df.isolated_x.apply(lambda x: np.isnan(x).all()))]

    # get rid of isolated dunes that have no slopes on one side due to segmentizing resolution
    mask = df.apply(lambda x: (x.isolated_height[0] == x.isolated_height[x.isolated_peak_idx]) or 
                    (x.isolated_height[-1] == x.isolated_height[x.isolated_peak_idx]), axis=1)
    df = df[~mask]
    
    df["left_fit"] = df.apply(lambda x: np.polyfit(x.isolated_x[:x.isolated_peak_idx+1], x.isolated_height[:x.isolated_peak_idx+1],deg=1), axis=1)
    df["right_fit"] = df.apply(lambda x: np.polyfit(x.isolated_x[x.isolated_peak_idx:], x.isolated_height[x.isolated_peak_idx:],deg=1), axis=1)
    
    df["left_ang_deg"] = df.left_fit.apply(lambda x: abs(np.arctan(x[0]))*180/np.pi)
    df["right_ang_deg"] = df.right_fit.apply(lambda x: abs(np.arctan(x[0]))*180/np.pi)
    
    # get the area from lower trough as slice and then calcualte volume per dune and then pper tile
    df["area_slice"] = df.apply(lambda x: get_isolated_area(x.isolated_x, x.isolated_height, x.isolated_peak_idx), axis=1)
    gdf_length = gdf_lines.length
    gdf_length.name = "length"
    volume_mean = pd.concat([df.area_slice.groupby(level=0).mean(),gdf_length],axis=1,join="inner").apply(lambda x: x["area_slice"]*x["length"],axis=1).mean()
    volume_median = pd.concat([df.area_slice.groupby(level=0).median(),gdf_length],axis=1,join="inner").apply(lambda x: x["area_slice"]*x["length"],axis=1).median()
    
    # new one weith the height of troughs and valleys
    return (row.linear_ang, df.left_ang_deg.values, df.right_ang_deg.values, 
            df.apply(lambda x: x.x_trans_height[x.lower_trough_idx],axis=1).values, df.apply(lambda x: x.trans_height[x.lower_trough_idx],axis=1).values,
            df.apply(lambda x: x.x_trans_height[x.peak_idx],axis=1).values , df.apply(lambda x: x.trans_height[x.peak_idx],axis=1).values, 
            df.apply(lambda x: x.x_trans_height[x.upper_trough_idx],axis=1).values, df.apply(lambda x: x.trans_height[x.upper_trough_idx],axis=1).values,
            volume_median,volume_mean)

def slope_hist(x,bins=1000,density=True):
    try:
        return np.histogram(x,bins=bins,density=density)
    except ValueError:
        return np.nan,np.nan

def diff_bmidl(x):
    try:
        return (x[:-1] + x[1:]) / 2
    except TypeError:
        return np.nan

def diff_hist_norm(diff_hist,diff_bedge):
    try:
        return diff_hist*np.diff(diff_bedge)
    except ValueError:
        return np.nan

def height_from_lower_trough(tlow,tpeak,thigh):
    """Return the minimum and maximum difference"""
    try:
        diff_min = []
        diff_max = []
        for t1,tp,t2 in zip(tlow,tpeak,thigh):
            diff1 = tp-t1
            diff2 = tp-t2
            if diff1>diff2:
                diff_min.append(diff2)
                diff_max.append(diff1)
            else:
                diff_min.append(diff1)
                diff_max.append(diff2)

        diff_mean = (np.array(diff_min)+np.array(diff_max))/2
        return float(np.nanmean(diff_min)), float(np.nanmean(diff_max)), float(np.nanmean(diff_mean))
    except TypeError:
        return np.nan

def dune_width(xlow,xhigh):
    try:
        diff_x = np.array(xhigh)-np.array(xlow)
        if len(diff_x)==0:
            return np.nan
        return np.nanmean(diff_x)
    except TypeError:
        return np.nan

def get_j_and_l(df,l_idx,thresh=100):

    prev_j = df.loc[l_idx][df.loc[l_idx].line_length<thresh].index_right.unique()
    prev_l = df[df.index_right.isin(prev_j)][df[df.index_right.isin(prev_j)].line_length<thresh].index.unique()

    while True:
        test_j = df.loc[prev_l][df.loc[prev_l].line_length<thresh].index_right.unique()
        test_l = df[df.index_right.isin(test_j)][df[df.index_right.isin(test_j)].line_length<thresh].index.unique()
        if (len(prev_j)==len(test_j)) and (len(prev_l),len(test_l)):
            break
        prev_j, prev_l = test_j, test_l
    return prev_j, prev_l

def change_line(row, centroid):
    coords = np.array(row.geometry.coords)
    if row.yj_points.equals(Point(row.geometry.coords[0])):
        new_line = np.concatenate([np.array(centroid.coords),coords])
    elif row.yj_points.equals(Point(row.geometry.coords[-1])):
        new_line = np.concatenate([coords,np.array(centroid.coords)])
    return LineString(new_line)

# The new interative yj cleaning of lines stuff
def connect_overlapping_lines(lines):
    if len(lines)==1:
        return lines[0]
    else:
        if lines[0].length > lines[1].length:
            return lines[0]
        else:
            return lines[1]
            
def test_func(lines):
    # line not altered by new yjunction (equal)
    if (len(lines)==1) or lines[0].equals(lines[1]):
        return lines[0]
    else:
        coords0 = lines[0].coords
        coords1 = lines[1].coords

        # both lines got an extra coordinate
        if (len(coords0) == len(coords1)):
            # coords1 got extra coordinate at the front coords0 at the end
            if coords0[0] == coords1[1]:
                out = np.concatenate([coords1,[coords0[-1]]])
            # coords0 got extra coordinate at the front coords1 at the end
            else:
                out = np.concatenate([coords0,[coords1[-1]]])
        elif (len(coords0) > len(coords1)):
            out = coords0
        else:
            out = coords1
                
        return LineString(out)

### NEW FUNCTIONS FOR CHUNKED ANGLE THING ###
def get_chunked_line(line,step):
    linel = line.length
    pnts = line.interpolate(np.arange(linel%step/2,linel-linel%step/2+step,step))
    pnts = [(pnt.x,pnt.y) for pnt in pnts]
    if len(pnts)==1:
        return MultiLineString([line])
    line_sects = []
    for i in range(len(pnts)):
        try:
            line_sects.append(LineString([[pnts[i][0],pnts[i][1]],[pnts[i+1][0],pnts[i+1][1]]]))
        except IndexError:
            continue
    return MultiLineString(line_sects)

def ang_sect(line):
    xs = np.asarray(line.xy[0])
    ys = np.asarray(line.xy[1])

    if len(np.unique(xs))>1:
        if len(xs)>2:
            linea0 = np.arctan(np.polyfit(xs,ys,deg=1)[0])
        else:
            linea0 = np.arctan((ys[1]-ys[0])/(xs[1]-xs[0]))
    else:
        linea0 = np.pi/2

    if len(np.unique(ys))>1:
        if len(ys)>2:
            linea1 = np.arctan(np.polyfit(ys,-xs,deg=1)[0])
        else:
            linea1 = np.arctan((xs[0]-xs[1])/(ys[1]-ys[0]))
    else:
        linea1 = np.pi/2

    return linea0,linea1
    
def ang_group(linea0,linea1):
    if circstd(linea1)>circstd(linea0):
        crestangleavgs = np.angle(np.sum(np.exp(1j*linea0)))
        crestanglestds = circstd(linea0)
    else:
        crestangleavgs = np.angle(np.sum(np.exp(1j*(linea1+np.pi/2))))
        crestanglestds = circstd(linea1)

    return crestangleavgs, crestanglestds

### new yj bisector things
def get_bisectors_for_line_test(pnt,line_series,radius=[0.51]):
    if len(line_series)<3:
        return np.ones(10)*np.nan
    yj = Yjunction(pnt,line_series)
    yj.get_radius_intersects(radius)
    yj.get_angle_intersects()
    yj.get_angle_between_points()
    # print(yj.get_angle_between_points())
    yj.get_bisectors_between_points()
    junction_params = yj.get_junction_opening_and_direction()
    junction_params = list(junction_params.values())[0]

    # output dictionary init
    outdict = {}
    outdict["minrad_oang"] = np.nan
    outdict["minrad_bdir"] = np.nan
    fillmax = False
    for i,rad in enumerate(radius):
        try:
            outdict[str(rad)+"_oang"] = junction_params[str(rad)]["opening_angle"]
            outdict[str(rad)+"_bdir"] = junction_params[str(rad)]["bisector_direction"]
        except KeyError:
            if i == 0:
                outdict["minrad_oang"] = list(junction_params.values())[0]["opening_angle"]
                outdict["minrad_bdir"] = list(junction_params.values())[0]["bisector_direction"]
            elif len(yj.radii)>1:
                fillmax = True
            outdict[str(rad)+"_oang"] = np.nan
            outdict[str(rad)+"_bdir"] = np.nan
            
    if fillmax:
        outdict["maxrad_oang"] = list(junction_params.values())[-1]["opening_angle"]
        outdict["maxrad_bdir"] = list(junction_params.values())[-1]["bisector_direction"]
    else:
        outdict["maxrad_oang"] = np.nan
        outdict["maxrad_bdir"] = np.nan

    return outdict

def nanmean(angles):
    mask = np.isnan(angles)
    if mask.all():
        return np.nan
    else:
        return np.nanmean(angles[~mask])
        
def circnanmean(angles):
    mask = np.isnan(angles)
    if mask.all():
        return np.nan
    else:
        return circmean(angles[~mask],high=np.pi,low=-np.pi)

def count_yjs(series):
    return series.value_counts()

def find_nearest_crest_points_new(row, cx_lines, cell_ang, cell_ang_flipped):
    bounds = cx_lines.bounds

    lon_min = bounds[0]
    lon_max = bounds[2]
    lat_min = bounds[1]
    lat_max = bounds[3]
    lat_width = abs(lat_max-lat_min)
    lon_width = abs(lon_max-lon_min)
    if (lat_width > lon_width):
        width = lat_width/2
    else:
        width = lon_width/2

    # since -pi =< ang < 0 adding half pi ensures to get transect left of the "orientation"
    ang = row.angs_3577+(np.pi/2)

    # check what would be left and right of the crest:
    angdiff = abs(cell_ang - row.angs_3577)
    if ((angdiff>(np.pi/2)) and (angdiff<(1.5*np.pi))):
        flippnts = True
    else:
        flippnts = False
    lon,lat = row.centroids.x,row.centroids.y

    #line1
    x1 = lon-np.cos(ang) * width
    y1 = lat-np.sin(ang) * width
    x1 = np.array([x1,lon])
    y1 = np.array([y1,lat])
    line1 = LineString(np.stack((x1,y1),axis=1))

    multipnt1 = cx_lines.intersection(line1)

    if multipnt1.is_empty:
        return Point()
        
    if isinstance(multipnt1,LineString):
        return Point()

    if isinstance(multipnt1,MultiLineString):
        return Point()
        
    if isinstance(multipnt1,GeometryCollection):
        multipnt1 = gpd.GeoSeries(multipnt1.geoms).explode()
        multipnt1 = multipnt1[multipnt1.geom_type=="Point"].values
        if len(multipnt1) == 0:
            return Point()
        elif len(multipnt1) == 1:
            multipnt1 = multipnt1[0]
        else:
            multipnt1 = MultiPoint(multipnt1)
        
    if isinstance(multipnt1, MultiPoint):
        multipnt1 = gpd.GeoSeries(list(multipnt1.geoms))
        multipnt1_dist = multipnt1.distance(row.centroids).sort_values()
        try:
            pnt1 = multipnt1.loc[multipnt1_dist[multipnt1_dist > 1].index[0]]
        except IndexError:
            return Point()
    elif isinstance(multipnt1, Point):
        # the single point that got identified is actually the centroid return empty
        if (abs(row.centroids.distance(multipnt1)) < 1):
            return Point()
        pnt1 = multipnt1
    
    #line2
    x2 = lon+np.cos(ang) * width
    y2 = lat+np.sin(ang) * width
    x2 = np.array([lon,x2])
    y2 = np.array([lat,y2])
    line2 = LineString(np.stack((x2,y2),axis=1))

    multipnt2 = cx_lines.intersection(line2)

    if multipnt2.is_empty:
        return Point()

    if isinstance(multipnt2,LineString):
        return Point()

    if isinstance(multipnt2,MultiLineString):
        return Point()
        
    if isinstance(multipnt2,GeometryCollection):
        multipnt2 = gpd.GeoSeries(multipnt2.geoms).explode()
        multipnt2 = multipnt2[multipnt2.geom_type=="Point"].values
        if len(multipnt2) == 0:
            return Point()
        elif len(multipnt2) == 1:
            multipnt2 = multipnt2[0]
        else:
            multipnt2 = MultiPoint(multipnt2)
            
    if isinstance(multipnt2, MultiPoint):
        multipnt2 = gpd.GeoSeries(list(multipnt2.geoms))
        multipnt2_dist = multipnt2.distance(row.centroids).sort_values()
        try:
            pnt2 = multipnt2.loc[multipnt2_dist[multipnt2_dist > 1].index[0]]
        except IndexError:
            return Point()
    elif isinstance(multipnt2, Point):
        # the single point that got identified is actually the centroid return empty
        if (abs(row.centroids.distance(multipnt2)) < 1):
            return Point()
        pnt2 = multipnt2

    if not flippnts:
        multipnt_out = MultiPoint([pnt2, pnt1])
    else:
        multipnt_out = MultiPoint([pnt1, pnt2])
        
    if cell_ang_flipped:
        multipnt_out = MultiPoint([multipnt_out.geoms[1],multipnt_out.geoms[0]])

    return multipnt_out
    
def find_shallower_trough_as_threshold(row):
    peak = row.trans_height[row.peak_idx]
    lower_trough = row.trans_height[row.lower_trough_idx]
    upper_trough = row.trans_height[row.upper_trough_idx]

    if (np.isnan(lower_trough) and np.isnan(upper_trough)):
        threshold = np.nan
    elif np.isnan(lower_trough):
        threshold = (peak-upper_trough)
    elif np.isnan(upper_trough):
        threshold = (peak-lower_trough)
    else:
        if lower_trough < upper_trough:
            threshold = (peak-upper_trough)
        else:
            threshold = (peak-lower_trough)

    threshold = peak-threshold
    return threshold

def get_slice_area(x,y):
    length = ((x[-1]-x[0])**2+(y[-1]-y[0])**2)**0.5
    m,b = np.polyfit(np.array([x[0],x[-1]]),np.array([y[0],y[-1]]),deg=1)
    yfit = m*x+b
    yfit[yfit<0]=0
    return (np.trapezoid(abs(y-yfit),x=x),np.nanmax(y-yfit),length)

def limit_to_shallow_trough(row):
    "limit to shallowest trough and return isolated for area"
    if (row.trans_height[row.lower_trough_idx] >= row.trans_height[row.upper_trough_idx]):
        mask = row.trans_height[row.lower_trough_idx:row.upper_trough_idx] >= row.trans_height[row.lower_trough_idx]
    else:
        mask = row.trans_height[row.lower_trough_idx:row.upper_trough_idx] >= row.trans_height[row.upper_trough_idx]
    return(row.x_trans_height[row.lower_trough_idx:row.upper_trough_idx][mask],row.trans_height[row.lower_trough_idx:row.upper_trough_idx][mask])
    
def handle_rasterio_new(row, cop_dict, samp_inter=500, samp_offset=50, pad=0.055):
    print("\r%05d"%(row.name), end="")
    cop_files = cops_idx_to_filename(row["COPs"],cop_dict)
    lons,lats,img = era5_pad_window(cop_files,row.lon,row.lat,pad)
    inter = RegularGridInterpolator((lats,lons), img, bounds_error=False, fill_value=np.nan, method="cubic") # lats first!
    
    try:
        gdf_lines = gpd.GeoDataFrame(geometry=list(row.clipped_lines_3577.geoms),crs=3577) 
    except AttributeError:
        gdf_lines = gpd.GeoDataFrame(geometry= [MultiLineString([row.clipped_lines_3577])],crs=3577)
    gdf_lines = gdf_lines[~((gdf_lines.geom_type=="Point") | (gdf_lines.geom_type=="MultiPoint"))] # to get rid of the points that are the product of the clipping
    df = pd.concat(gdf_lines.geometry.apply(lambda x: samp_line(x,samp_inter,samp_offset)).to_list(), axis=0) # determines the spacing and offset of sampling THIS CAN BE altered!!!
    df["group"] = (df.index == 0).cumsum() - 1
    index_dict = dict(enumerate(list(gdf_lines.index),0))
    df['index'] = df.group.map(index_dict)
    df = df.set_index(['index', df.groupby('index').cumcount()])
    #TESTING
    # return df
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="invalid value encountered in intersection")
        df["neighbours"] = df.apply(lambda x: find_nearest_crest_points_new(x, row.geometry_3577, row.linear_ang, row.avg100sig >= 0),axis=1) #potential timesafe
    # THIS PART HERE MIGHT NEED SO FURTHWER THINKING TO CHEKC THAT STEPS ARE NOT IGNORED BECAUS OF MISSING DUNE CRESTS ON EITHER SIDE MATYBE ONE SIDE IS GOODF?
    df = df[~gpd.GeoSeries(df.neighbours).is_empty]
    if df.empty:
        return row.linear_ang, np.nan, np.nan
    df["trans_3577"] = df.apply(lambda x: LineString([[x.neighbours.geoms[0].x,x.neighbours.geoms[0].y],
                                                      [x.centroids.x, x.centroids.y],
                                                      [x.neighbours.geoms[1].x,x.neighbours.geoms[1].y]]).segmentize(1),axis=1) # potential timesafe
    df["trans_4326"] = gpd.GeoSeries(df.trans_3577).set_crs(3577).to_crs(4326)
    df["centroid_idx_3577"] = df.apply(lambda x: find_centroid_idx(x.trans_3577,x.centroids), axis=1)
    df["trans_height"] = df.trans_4326.apply(lambda x: inter((x.xy[1],x.xy[0])))
    dunes_idxs = df.apply(lambda x: get_peak_troughs(x.trans_height, x.name, x.centroid_idx_3577),axis=1)
    df[["lower_trough_idx","peak_idx","upper_trough_idx"]] = pd.DataFrame(dunes_idxs.to_list(),index=df.index)
    df = df[~(df.peak_idx==-1)] # get rid of transects that have no peaks
    if df.empty:
        return row.linear_ang, np.nan, np.nan
    # TESTING
    # return df
    # this is to get the half height from shallowest point aand to get the slope at half the height
    isolated_dune = df.apply(lambda x: isolate_dune(x.trans_3577, x.trans_height,
                                                      x.lower_trough_idx, x.peak_idx, 
                                                      x.upper_trough_idx, relative_height_thresh(x)), axis=1)
    df[["x_trans_height", "isolated_x", "isolated_height", "isolated_peak_idx"]] = pd.DataFrame(isolated_dune.to_list(),index=df.index)
    df = df[~(df.isolated_x.apply(lambda x: np.isnan(x).all()))]
    
    # get rid of isolated dunes that have no slopes on one side due to segmentizing resolution
    mask = df.apply(lambda x: (x.isolated_height[0] == x.isolated_height[x.isolated_peak_idx]) or 
                    (x.isolated_height[-1] == x.isolated_height[x.isolated_peak_idx]), axis=1)
    df = df[~mask]

    #slopes
    df["left_fit"] = df.apply(lambda x: np.polyfit(x.isolated_x[:x.isolated_peak_idx+1], x.isolated_height[:x.isolated_peak_idx+1],deg=1), axis=1)
    df["right_fit"] = df.apply(lambda x: np.polyfit(x.isolated_x[x.isolated_peak_idx:], x.isolated_height[x.isolated_peak_idx:],deg=1), axis=1)
    df["left_ang_deg"] = df.left_fit.apply(lambda x: abs(np.arctan(x[0]))*180/np.pi)
    df["right_ang_deg"] = df.right_fit.apply(lambda x: abs(np.arctan(x[0]))*180/np.pi)
    df["mean_slope"] = df.apply(lambda x: np.nanmean([x.left_ang_deg,x.right_ang_deg]),axis=1)
    df["left_minus_right_ang"] = df["left_ang_deg"] - df["right_ang_deg"]

    # this here is to get rid of some problem where troughs are actually nan! this must be becuase of a missing trought on one side and my
    # code "get_peak_trough" above returning the len -1 idx for these cases! This could potentially be a huge problem when calcualting the volume
    # need to look at output in order to actually say something about that but it would lead to dunes be overestimated! this is why i added shallow
    nan_at_trough_mask = df.apply(lambda x: np.isnan(x.trans_height[x.lower_trough_idx]) | np.isnan(x.trans_height[x.upper_trough_idx]),axis=1)
    df = df[~nan_at_trough_mask]

    # get area shallow trough (closer to peak)
    area_height_width_shallow = df.apply(lambda x: get_slice_area(*limit_to_shallow_trough(x)),axis=1)
    df[["slice_area_shallow","rel_height_shallow","width_shallow"]] = pd.DataFrame(area_height_width_shallow.to_list(),index=df.index)

    # get area all
    area_height_width = df.apply(lambda row: get_slice_area(row.x_trans_height[row.lower_trough_idx:row.upper_trough_idx],
                                                     row.trans_height[row.lower_trough_idx:row.upper_trough_idx]),axis=1)
    df[["slice_area","rel_height","width"]] = pd.DataFrame(area_height_width.to_list(),index=df.index)
    
    gdf_length = gdf_lines.length
    gdf_length.name = "length"
    volume = pd.concat([df.slice_area.groupby(level=0).median(),gdf_length],axis=1,join="inner")
    volume = (volume["length"]*volume["slice_area"])
    volume_sum = volume.sum()
    volume_median = volume.median()
    volume_mean = volume.mean()

    volume_shallow = pd.concat([df.slice_area_shallow.groupby(level=0).median(),gdf_length],axis=1,join="inner")
    volume_shallow = (volume_shallow["length"]*volume_shallow["slice_area_shallow"])
    volume_shallow_sum = volume_shallow.sum()
    volume_shallow_median = volume_shallow.median()
    volume_shallow_mean = volume_shallow.mean()

    df["max_height"] = df.apply(lambda x: np.where(x.trans_height[x.lower_trough_idx]<=x.trans_height[x.upper_trough_idx],
                                     x.trans_height[x.peak_idx]-x.trans_height[x.lower_trough_idx],
                                     x.trans_height[x.peak_idx]-x.trans_height[x.upper_trough_idx]),axis=1).astype(float)
    #TESTING
    # return df
    
    return np.concatenate([df.loc[:,['left_ang_deg', 'right_ang_deg',"mean_slope", "left_minus_right_ang", 'rel_height', 'width',"rel_height_shallow","width_shallow","max_height"]].mean().values,
                           df.loc[:,['left_ang_deg', 'right_ang_deg',"mean_slope", "left_minus_right_ang", 'rel_height', 'width',"rel_height_shallow","width_shallow","max_height"]].median().values,
                           np.array([volume_sum]),np.array([volume_mean]),np.array([volume_median]),
                           np.array([volume_shallow_sum]),np.array([volume_shallow_mean]),np.array([volume_shallow_median])])

# Calculation

### Step 0: Merge lines

After extracting crestlines from previous steps two lines that share the same end points are not treated as one continous line. The below fixes this.

In [ ]:
%%time
crestlines_SKL = gpd.read_file("../Data/Crestlines/SKL_combined.shp")
gpd.GeoDataFrame(geometry=list(line_merge(MultiLineString(crestlines_SKL.geometry.values)).geoms),crs=crestlines_SKL.crs).to_file("../Data/Crestlines/SKL_merged.shp")

### Step 1: Crop SKL crestlines to GSA cells

Reduce number cells with "crest lines" to only cells that have been identified as dunes. Geoscience Australia sandridges dataset was used (GEODATA TOPO 250K Series 3 sand ridges dataset is available from Geoscience Australia (https://pid.geoscience.gov.au/dataset/ga/64058)). Some manual cells where added and others removed to only focus on (mostly) dune containing cells.

In [ ]:
%%time
save=False
res  = 0.1
lonlatbounds = [[112.9,153.4],[-38.9,-10.5]]

# Era5-Land coordinates
lon = np.round(np.arange(lonlatbounds[0][0],lonlatbounds[0][1]+1e-4,res),1)
lat = np.round(np.arange(lonlatbounds[1][0],lonlatbounds[1][1]+1e-4,res),1)

# GSA and SKL crestlines 
crestlines  = gpd.read_file('../Data/Crestlines/sandridges.shp') # from Geoscience Australia (GSA)
crestlines_SKL = gpd.read_file("../Data/Crestlines/SKL_merged.shp")

# manually added cells (points in cell) to not crop them by GSA
gdf_missed_GSA = gpd.read_file("../Data/Crestlines/missed_GSA_cell_points.shp") # file created from manual inspection where dunes exist but are not include in GSA lines

# cells to delete from the signing step further down
gdfmaskremove = gpd.read_file("../Data/Crestlines/SKL_rect_delete_points.shp") # file created from manual inspection where cell is not representing dunes (e.g. rivers or mountains masking dune signal)

# Get lat and lon index
lt_ln = np.array([(round(lt,1),round(ln,1)) for lt in lat for ln in lon])
lt_ln_idx = np.array([(ltidx,lnidx) for ltidx in range(len(lat)) for lnidx in range(len(lon))])

# Create rectangle polygons around ERA5-land 
polygons = [ 
    Polygon([
        (round(ln - res/2,2), round(lt - res/2,2)),
        (round(ln + res/2,2), round(lt - res/2,2)),
        (round(ln + res/2,2), round(lt + res/2,2)),
        (round(ln - res/2,2), round(lt + res/2,2)),
        (round(ln - res/2,2), round(lt - res/2,2))  # Close the polygon
    ])
    for lt in lat for ln in lon
]

# Create a polygon GeoDataFrame and make it match the crs one from the GSA lines
gdf_rect = gpd.GeoDataFrame(geometry=polygons, crs="EPSG:4326").to_crs(crestlines.crs)
gdf_rect["lat_idx"] = lt_ln_idx[:,0]
gdf_rect["lon_idx"] = lt_ln_idx[:,1]
gdf_rect["lat"] = lt_ln[:,0]
gdf_rect["lon"] = lt_ln[:,1]
missed_rects = gdf_rect.loc[gpd.sjoin(gdf_rect,gdf_missed_GSA.to_crs(gdf_rect.crs)).index].copy()

# Duplicate ERA5-land cell of gdf_rect geometry and spatial join with GSA crestlines
gdf_rect['geom_poly'] = gdf_rect.geometry
gdf = gpd.sjoin(crestlines, gdf_rect,how="inner",predicate="intersects",lsuffix="left",rsuffix="right")
gdf = gdf.rename({"index_right":"index_poly"},axis=1).loc[:,"geometry":].explode(index_parts=False)

# Update gdf_rect to only contain the ERA5 cells that intersect with GSA crests
gdf_rect = gdf.set_geometry("geom_poly").drop("geometry",axis=1).rename_geometry("geometry")
gdf_rect["geom_poly"] = gdf_rect.geometry 
gdf_rect = gdf_rect.groupby("index_poly").agg("first")
gdf_rect = gdf_rect.set_crs(gdf.crs)

# add missing cells
gdf_rect = pd.concat([gdf_rect,missed_rects]).sort_index()
gdf_rect["geom_poly"] = gdf_rect.geometry
gdf_rect.index.name="index_poly"

idx_to_remove = gpd.sjoin(gdf_rect,gdfmaskremove.to_crs(gdf_rect.crs)).index.unique()
gdf_rect = gdf_rect[~gdf_rect.index.isin(idx_to_remove)]

# SKL crestlines cropped by era5 cells that intersect with GSA crestlines
gdf_rect = gdf_rect.to_crs(crestlines_SKL.crs)
gdf = gpd.sjoin(crestlines_SKL, gdf_rect, how="inner",predicate="intersects",lsuffix="left",rsuffix="right")
gdf = gdf.drop_duplicates(subset=['geometry'])

# update the rect gdf
gdf_rect = gdf.set_geometry("geom_poly").loc[:,"index_poly":].rename_geometry("geometry").groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first","geometry":"first"}).set_geometry("geometry").set_crs(gdf.crs)

if save:
    # save the ERA5 cells and the cropped lines without duplicates (caused by the spatial join)
    gdf.geometry.to_file("../Data/Crestlines/SKL_merged_cropped.shp") # save the cropped crestlines
    gdf_rect.to_file("../Data/Crestlines/SKL_rect.shp", index=True) # save the cropped cell mask

### Step 2: Junctions and update lines

Semi-automatic approach to calculate the y-junctions from lines, remove and reconnect y-junctions that are shorter than 100 m to account for raster induced artifacts and recalculate the junctions and lines.

#### Step 2a: Get Junctions and junction lines from original lines

In [ ]:
%%time
lines = gpd.read_file("../Data/Crestlines/SKL_merged_cropped.shp")
lines.drop(lines.columns[0],axis=1,inplace=True)

gdf_all, gdf_yj, gdf_multi, gdf_terms =  get_junctions(lines)

# fix to get rid of terminations that are rings (since a ring still has ends)
gdf_terms.sort_index(inplace=True)
gdf_terms.reset_index(drop=True, inplace=True)
termsjoin = gpd.sjoin(gdf_terms,gdf_terms)
termsjoin = (termsjoin[termsjoin.index!=termsjoin.index_right])
gdf_terms = gdf_terms[~gdf_terms.index.isin(termsjoin.index)]

gdf_yj.J_points.set_crs(gdf_yj.crs).to_file("../Data/Crestlines/y_junction_points.shp",index=True)
gdf_yj.geometry.to_file("../Data/Crestlines/y_junction_lines.shp",index=True)
gdf_multi.J_points.set_crs(gdf_multi.crs).to_file("Shapefiles/multi_junction_points.shp",index=True)
gdf_multi.geometry.to_file("../Data/Crestlines/multi_junction_lines.shp",index=True)
gdf_terms.geometry.to_file("../Data/Crestlines/terminations.shp",index=True)
gdf_all.J_points.set_crs(gdf_all.crs).to_file("../Data/Crestlines/all_junctions.shp",index=True)

#### Step 2b: Remove short lines from junctions and update lines 

In [ ]:
# set line length threshold (in meter)
thresh=100

# yj points
yj = gpd.read_file("../Data/Crestlines/y_junction_points.shp")
yj.set_index("Index", inplace=True)

# lines
gdf = gpd.read_file("../Data/Crestlines/SKL_merged_cropped.shp")
gdf.set_index("FID", inplace=True)

SKL_in = gpd.sjoin(gdf,yj,how="inner",predicate="intersects",lsuffix="left",rsuffix="right")
SKL_in.rename(columns={"Index":"index_right"}, inplace=True)
SKL_in["line_length"] = SKL_in.to_crs(3577).length.values
SKL_in.loc[:,"yj_points"] = yj.loc[SKL_in.index_right].geometry.values

# just taking the short duplicates
unique_idx = (SKL_in.loc[SKL_in.index[SKL_in.index.duplicated()]][SKL_in.loc[SKL_in.index[SKL_in.index.duplicated()]].line_length<=thresh]).index.unique()
SKL_out = SKL_in.set_index([SKL_in.index,"index_right"]) 

In [ ]:
%%time
idx_to_delete = None
for i,l in enumerate(unique_idx):
    print("\r%7d/%7d %7d"%(i+1,len(unique_idx),l), end='')
    if not idx_to_delete is None:
        if (l in idx_to_delete.get_level_values(0)):
            continue
    try:
        test_j, test_l = get_j_and_l(SKL_in,l,thresh)
    except KeyError:
        continue
    if len(test_j)==0:
        continue

    #get lines that will make up real junction (longer ones)
    dftemp = SKL_in[SKL_in.index_right.isin(test_j)]
    dftemp = dftemp[dftemp.line_length>thresh]
    
    # special case
    if ((len(dftemp.index_right.unique())==2) and (len(dftemp.index.unique())==3)):
        if not ((dftemp.is_ring).any() or len(dftemp)==4):
            dupidx = dftemp.index_right.value_counts().index[0]
            sigidx = dftemp.index_right.value_counts().index[1]
            ilocidx = np.argwhere(dftemp.index_right==sigidx)[0][0]
            line_to_change = dftemp.iloc[ilocidx].geometry
            former_line_yj = dftemp.iloc[ilocidx].yj_points
            centtemp = dftemp.yj_points[dftemp.index_right==dupidx].iloc[0]
            if(former_line_yj.coords[0] == line_to_change.coords[0]):
                linetemp = LineString(np.concatenate([np.array([centtemp.coords[0]]),line_to_change.coords[:]]))
            else:
                linetemp = LineString(np.concatenate([line_to_change.coords[:],np.array([centtemp.coords[0]])]))
            dftemp.iloc[ilocidx,0] = linetemp
        
    # only connect open ends of three in center rest left unconnected
    # elif not (len(dftemp)>3) and not dftemp.empty:
    elif not (len(dftemp)>3) and not dftemp.empty and not (len(dftemp.index.unique())==2): # prevents loop overlaps (hoepfully)
        # centtemp = dftemp.yj_points.values.unary_union().centroid
        centtemp = unary_union(dftemp.yj_points.values).centroid
        dftemp["geometry"] = dftemp.apply(lambda row: change_line(row,centtemp),axis=1)

    # create multiindices to allow assigning values
    dftemp.set_index([dftemp.index,"index_right"],inplace=True)
    
    # update the lines that will be kept
    SKL_out.loc[dftemp.index,"geometry"] = dftemp.geometry
    
    # gather indices to delete
    mask = SKL_in.index.isin(test_l) & SKL_in.index_right.isin(test_j)
    mith_mask = SKL_in.loc[mask]

    mith_mask.set_index([mith_mask.index,"index_right"],inplace=True)
    if idx_to_delete is None:
        idx_to_delete = mith_mask.index
    else:
        idx_to_delete = idx_to_delete.append(mith_mask.index)

In [ ]:
%%time
SKL_out_clean = SKL_out.drop(idx_to_delete)
SKL_out_clean = SKL_out_clean.reset_index().groupby("FID").agg({"geometry" : lambda x: test_func(x.geometry.values)})
SKL_out_clean.set_geometry("geometry",inplace=True,crs=SKL_in.crs)

idx_to_add = gdf.index[~gdf.index.isin(SKL_in.index.unique())]
SKL_out_clean = pd.concat([SKL_out_clean,gdf.loc[idx_to_add,["geometry"]]])

# SKL_out_clean.to_file("Testing/SKL_merged_cropped_clean_%d.shp"%(thresh))

# SKL_out_clean_merge=gpd.GeoDataFrame(geometry=list(line_merge(MultiLineString(SKL_out_clean.geometry.values)).geoms),crs=SKL_out_clean.crs)
# SKL_out_clean_merge.to_file("Testing/SKL_merged_cropped_clean_%d_merge.shp"%(thresh))

SKL_out_clean.to_file("../Data/Crestlines/SKL_merged_cropped_clean_%d.shp"%(thresh))

SKL_out_clean_merge=gpd.GeoDataFrame(geometry=list(line_merge(MultiLineString(SKL_out_clean.geometry.values)).geoms),crs=SKL_out_clean.crs)
SKL_out_clean_merge.to_file("../Data/Crestlines/SKL_merged_cropped_clean_%d_merge.shp"%(thresh))

#### Step 2c:  Get Junctions and junction lines from updated lines

In [ ]:
%%time
lines = gpd.read_file("../Data/Crestlines/SKL_merged_cropped_clean_%d_merge.shp"%(thresh))
lines.drop(lines.columns[0],axis=1,inplace=True)

gdf_all, gdf_yj, gdf_multi, gdf_terms =  get_junctions(lines)

# fix to get rid of terminations that are rings (since a ring still has ends)
gdf_terms.sort_index(inplace=True)
gdf_terms.reset_index(drop=True, inplace=True)
termsjoin = gpd.sjoin(gdf_terms,gdf_terms)
termsjoin = (termsjoin[termsjoin.index!=termsjoin.index_right])
gdf_terms = gdf_terms[~gdf_terms.index.isin(termsjoin.index)]

gdf_yj.J_points.set_crs(gdf_yj.crs).to_file("../Data/Crestlines/y_junction_points_updated.shp",index=True)
gdf_yj.geometry.to_file("../Data/Crestlines/y_junction_lines_updated.shp",index=True)
gdf_multi.J_points.set_crs(gdf_multi.crs).to_file("../Data/Crestlines/multi_junction_points_updated.shp",index=True)
gdf_multi.geometry.to_file("../Data/Crestlines/multi_junction_lines_updated.shp",index=True)
gdf_terms.geometry.to_file("../Data/Crestlines/terminations_updated.shp",index=True)
gdf_all.J_points.set_crs(gdf_all.crs).to_file("../Data/Crestlines/all_junctions_updated.shp",index=True)

##### Step 2c: Manual fix and rerun

A few lines have weird overlapping behaviour. And result in weird artifacts so points in lines were removed by hand. This means updating the SKL_merged above

In [ ]:
print("Fixing these by hand in QGIS")
print(gdf_yj[(gdf_yj.geometry.geom_type=="LineString")].index.values)
print("File saved will have '_manual' suffix. Load below")

Rerun to get the junctions

In [ ]:
%%time
lines = gpd.read_file("../Data/Crestlines/SKL_merged_cropped_clean_%d_merge_manual.shp"%(thresh))
lines.drop(lines.columns[0],axis=1,inplace=True)

gdf_all, gdf_yj, gdf_multi, gdf_terms =  get_junctions(lines)

# fix to get rid of terminations that are rings (since a ring still has ends)
gdf_terms.sort_index(inplace=True)
gdf_terms.reset_index(drop=True, inplace=True)
termsjoin = gpd.sjoin(gdf_terms,gdf_terms)
termsjoin = (termsjoin[termsjoin.index!=termsjoin.index_right])
gdf_terms = gdf_terms[~gdf_terms.index.isin(termsjoin.index)]

# fix to remove double rings from yj file and add to multi junctions
gdf_yj_explode = gdf_yj.explode()
idx_2l = gdf_yj_explode.index.value_counts()[gdf_yj_explode.index.value_counts()==2].index
gdf_yj_explode_2l = gdf_yj_explode[gdf_yj_explode.index.isin(idx_2l)]
idx_double_ring = (gdf_yj_explode_2l[gdf_yj_explode_2l.is_ring].index.value_counts(sort=False)==2)
idx_double_ring = idx_double_ring[idx_double_ring].index
gdf_multi = pd.concat([gdf_multi,gdf_yj.loc[idx_double_ring]])
gdf_yj = gdf_yj.drop(idx_double_ring)

# save all outputs
gdf_yj.J_points.set_crs(gdf_yj.crs).to_file("../Data/Crestlines/y_junction_points_updated_manual.shp",index=True)
gdf_yj.geometry.to_file("../Data/Crestlines/y_junction_lines_updated_manual.shp",index=True)
gdf_multi.J_points.set_crs(gdf_multi.crs).to_file("../Data/Crestlines/multi_junction_points_updated_manual.shp",index=True)
gdf_multi.geometry.to_file("../Data/Crestlines/multi_junction_lines_updated_manual.shp",index=True)
gdf_terms.geometry.to_file("../Data/Crestlines/terminations_updated_manual.shp",index=True)
gdf_all.J_points.set_crs(gdf_all.crs).to_file("../Data/Crestlines/all_junctions_updated_manual.shp",index=True)

### Step 3: Angle per ERA5 cell

Obtain unsigned cell-average dune orientation (circular mean and standard deviation) -180 to 0 degrees

#### Step 3a: Chunked line angles (new)

In [ ]:
%%time
# lines
gdf = gpd.read_file("../Data/Crestlines/SKL_merged_cropped_clean_100_merge_manual.shp") # change if needed
gdf.set_index("FID", inplace=True)

gdf_rings = gdf[gdf.is_ring]
gdf_no_rings = gdf[~gdf.is_ring].copy()

# ERA5 cells
gdf_rect = gpd.read_file("../Data/Crestlines/SKL_rect.shp")

gdf_no_rings.loc[:,"lines_3577"] = gdf_no_rings.to_crs(3577).geometry

step = 100 # in meter to resample along line
gdf_no_rings.loc[:,"lines_3577_sects"] = gdf_no_rings.apply(lambda x: get_chunked_line(x.lines_3577,step),axis=1)
gdf_no_rings_exp = gdf_no_rings.set_geometry("lines_3577_sects", crs=3577).explode(index_parts=True)
gdf_no_rings_exp.loc[:,["linea0","linea1"]] = gdf_no_rings_exp.apply(lambda x: pd.Series(ang_sect(x.lines_3577_sects)), axis=1).to_numpy()
# gdf_no_rings_exp.loc[:,"lines_3577_sects":].rename_geometry("geometry").to_file("../Data/Crestlines/SKL_merged_cropped_clean_merge_manual_sects_%d.shp"%(step))
gdf_no_rings_exp = gdf_no_rings_exp.loc[:,"lines_3577_sects":].rename_geometry("geometry")
gdf_no_rings_exp.reset_index(inplace=True)

#### Step 3b: Group angles (new)

In [ ]:
%%time
step = 100 # make sure step aligns with loaded file sects
# get chunked lines with angle from above 
# gdf_no_rings_exp = gpd.read_file("../Data/Crestlines/SKL_merged_cropped_clean_merge_manual_sects_%d.shp"%(step))
gdf_no_rings_exp.set_index([x for x in gdf_no_rings_exp.columns[[0,1]]],inplace=True)
gdf_no_rings_exp.rename_geometry("lines_3577_sects",inplace=True)

# get the  era5cells
gdf_rect = gpd.read_file("../Data/Crestlines/SKL_rect.shp")
gdf_rect.set_index(gdf_rect.columns[0], inplace=True)
gdf_rect["geom"] = gdf_rect.geometry

# transform the sects into 4326 for rect join
gdf_no_rings_exp["lines_4326_sects"] = gdf_no_rings_exp.lines_3577_sects.to_crs(gdf_rect.crs)

# get lines per cell join
gdf_no_rings_cell = gpd.sjoin(gdf_no_rings_exp.set_geometry("lines_4326_sects"), gdf_rect,
                how="inner", 
                predicate="intersects",
                lsuffix="left",
                rsuffix="right")

gdf_no_rings_cell = gdf_no_rings_cell.set_index("index_poly").groupby("index_poly").apply(lambda x: pd.Series(ang_group(x.linea0,x.linea1))).rename(columns={0:"linear_ang",1:"linear_std"})
gdf_no_rings_cell["linear_ang"] = gdf_no_rings_cell.linear_ang.apply(limit_neg_pi_less_0)
gdf_no_rings_cell = pd.concat([gdf_rect.loc[:,["lat","lon","lat_idx","lon_idx"]],gdf_no_rings_cell], axis=1)

points_3577 = gpd.GeoSeries(gdf_no_rings_cell.apply(lambda x: Point(x.lon,x.lat),axis=1)).set_crs(4326).to_crs(3577).apply(lambda x: (x.x,x.y))
gdf_no_rings_cell[["lon_3577","lat_3577"]] = pd.DataFrame(points_3577.tolist(), index=points_3577.index)
gdf_no_rings_cell["avg_crest_3577"] = gdf_no_rings_cell.apply(lambda x: generate_linestring(x.lon_3577,x.lat_3577,x.linear_ang,180*30),axis=1)
gdf_no_rings_cell = gdf_no_rings_cell.set_geometry("avg_crest_3577").set_crs(3577)
gdf_no_rings_cell.to_file("../Data/Crestlines/Average_angle_3577_from_sects_%d.shp"%(step),index=True)

### Step 4: Calculate median and mean spacing

Calculate the median and mean crest spacing for each cell

In [ ]:
%%time
lines = gpd.read_file("../Data/Crestlines/SKL_merged_cropped_clean_100_merge_manual.shp")
lines.drop(lines.columns[0],axis=1,inplace=True)

gdf_rect = gpd.read_file("../Data/Crestlines/SKL_rect.shp")
gdf_rect.set_index(gdf_rect.columns[0], inplace=True)

gdf_ang = gpd.read_file("../Data/Crestlines/Average_angle_3577_from_sects_100.shp")
gdf_ang.set_index(gdf_ang.columns[0], inplace=True)
gdf_ang = gdf_ang[gdf_ang.geometry!=None] # deal with empty average because of line end not considered in calculation in step 3 (57658 cell) only one small end in cell, out of chunk
gdf_ang["lat"] = gdf_ang.lat_3577
gdf_ang["lon"] = gdf_ang.lon_3577
gdf_ang.drop(["lon_3577","lat_3577"], axis=1, inplace=True)
gdf_ang["geometry_poly_intersect"] = gpd.sjoin(lines, gdf_rect,
                                            how="inner",
                                            predicate="intersects",
                                            lsuffix="left",
                                            rsuffix="right").to_crs(gdf_ang.crs).groupby("index_poly").agg({"geometry":lambda x: MultiLineString(x.to_list())})
trans_width = 10800

gdf_ang["geom_poly"] = gdf_rect.geometry.to_crs(gdf_ang.crs)
gdf_ang["transects"] = gdf_ang.apply(lambda x: generate_transects(x, trans_width,50), axis=1)

gdf_ang.transects.set_crs(gdf_ang.crs).to_file("../Data/Crestlines/Transects_3577_from_sects_100.shp", index=True)

temp_split = gdf_ang.set_geometry("transects").explode(index_parts=True).apply(lambda x: split(x.transects, x.geometry_poly_intersect).geoms[1:-1], axis=1).set_crs(gdf_ang.crs)
temp_split = temp_split[~temp_split.is_empty].explode(index_parts=True)
temp_split = temp_split[~temp_split.is_empty].explode(index_parts=True)
temp_split = temp_split[~temp_split.intersects(gdf_ang.geom_poly.exterior,align=True) & temp_split.within(gdf_ang.geom_poly,align=True)]
temp_split_3577 = temp_split.to_crs(3577)

splits = temp_split_3577.reset_index().set_index("index_poly").iloc[:,-1].groupby(level=0).agg(lambda x: MultiLineString(x.to_list()))
no_lines = splits.apply(lambda x: len(x.geoms))
cum_length = splits.apply(lambda x: x.length)

# create a dataframe with the transects line split by the dune lines, number of lines, cum length, mean and median spacing
spacings_df = pd.concat([splits.set_crs(3577).to_crs(gdf_ang.crs),no_lines,cum_length],axis=1)
spacings_df.columns = ["segs","no_segs","cum_segs"]
spacings_df["spc_med"] = temp_split_3577.length.groupby(level=0).agg('median') # use median
spacings_df["spc_mea"] = temp_split_3577.length.groupby(level=0).agg('mean')


spacings_df.set_geometry("segs").set_crs(temp_split.crs).to_file("../Data/Crestlines/Spacing_3577_from_sects_100.shp", index=True)
gdf_ang.geometry_poly_intersect.set_crs(gdf_ang.crs).to_file("../Data/Crestlines/Lines_per_cell_3577_from_sects_100.shp", index=True)

### Step 5: Interaction

Calculate the dune interaction points using the above junctions, cells per cell and the workflow by Marvin et al. 2023

In [ ]:
def get_interaction_points_new(row, crs, junctions):
    
    gdf = gpd.sjoin(row, row, how='inner',predicate="intersects",lsuffix="left",rsuffix="right")
    gdf = gdf[gdf.index != gdf.index_right]
    gdf["intersection_geom"] = gdf['geom_left'].intersection(gdf['geom_right'])
    gdf_old_geo_name = gdf.active_geometry_name
    gdf = gdf.set_geometry('intersection_geom', crs=row.crs).drop(gdf_old_geo_name,axis=1).rename_geometry(gdf_old_geo_name)
    gdf =  gdf.drop_duplicates(subset=['geometry'])["geometry"]

    if gdf.empty:
        return
    
    try:
        gdf = gpd.GeoDataFrame(geometry=list(gdf.union_all().geoms),crs=row.crs)
    except AttributeError:
        # handles the one polygon case of an union
        gdf = gpd.GeoDataFrame(geometry=[gdf.union_all()],crs=row.crs)
    # weird fix for single interaction point by duplicating the single point
    if len(gdf)==1:
        gdf["interaction_point"] = pd.concat([gdf,gdf]).centroid.to_crs(crs).iloc[[0]]
    else:
        gdf["interaction_point"] = gdf.centroid.to_crs(crs)
    gdf.to_crs(crs, inplace=True)

    #exceutes when there are no junctions to compare
    if junctions is None:
        gdf_old_geo_name = gdf.active_geometry_name
        gdf = gdf[gdf.interaction_point.intersects(row.geom_poly.iloc[0])].set_geometry("interaction_point").drop(columns=gdf_old_geo_name)
        # gdf = gdf[gdf.interaction_point.intersects(row.geom_poly.iloc[0])].set_geometry("interaction_point").drop(gdf_old_geo_name,axis=1)#.rename_geometry("interaction_point")
        return gdf

    intersect_poly_junctions_join = gpd.sjoin(gdf,junctions)
    junctions_idx = intersect_poly_junctions_join.index
    not_junctions_idx = gdf.index.difference(junctions_idx)
    gdf_old_geo_name = gdf.active_geometry_name
    gdf = gdf.loc[not_junctions_idx].set_geometry("interaction_point").drop(gdf_old_geo_name,axis=1)#.rename_geometry("interaction_point")
    junctions_interact_points = junctions.loc[intersect_poly_junctions_join.index_right.values].rename_geometry("interaction_point")
    gdf = pd.concat([gdf,junctions_interact_points])
    gdf = gdf[gdf.interaction_point.intersects(row.geom_poly.iloc[0])]

    print("\r%05d"%(row.iloc[0,0]), end='')
    
    return gdf

In [ ]:
%%time
gdf = gpd.read_file("../Data/Crestlines/Lines_per_cell_3577_from_sects_100.shp")
gdf.set_index(gdf.columns[0], inplace=True)
gdf.to_crs(4326,inplace=True)
spacing = gpd.read_file("../Data/Crestlines/Spacing_3577_from_sects_100.shp")
spacing.set_index(spacing.columns[0], inplace=True)
gdf_rect = gpd.read_file("../Data/Crestlines/SKL_rect.shp")
gdf_rect.set_index(gdf_rect.columns[0], inplace=True)

gdf = spacing.spc_med.to_frame().merge(gdf, on="index_poly", how="left")
gdf = gdf.set_geometry("geometry")
gdf = gdf.merge(gdf_rect.rename_geometry("geom_poly"), on="index_poly", how="left")

all_junctions = gpd.read_file("../Data/Crestlines/all_junctions_updated_manual.shp")
all_junctions.drop(all_junctions.columns[0],axis=1,inplace=True)

# get all the buffers around crestlines
buffer = gdf.explode(index_parts=True)
buffer["geometry"] = buffer.to_crs(3577).buffer(buffer.spc_med*(0.1/2))
buffer["geom"] = buffer.geometry
buffer.drop("spc_med", axis=1, inplace=True)

# get junctions per cell
gdf_junc_buff = gpd.sjoin(all_junctions, gdf_rect,
                how="inner",
                predicate="intersects",
                lsuffix="left",
                rsuffix="right")
gdf_junc_buff = gdf_junc_buff.groupby("index_poly").agg({'geometry':lambda x: MultiPoint(x.tolist())}).set_geometry("geometry").set_crs(all_junctions.crs).explode(index_parts=True).rename_geometry("J_points")
print(buffer.index.get_level_values(0)[-1])
buffer_interact = buffer.groupby(level=0).apply(lambda x: get_interaction_points_new(x.reset_index(), 
                                                                                 gdf.crs, 
                                                                                 check_junction_presence(gdf_junc_buff, x.index.get_level_values(0).values[0])))
buffer_interact.to_file("../Data/Crestlines/Interaction_point_sects_100.shp", index=True)

### Step 6: Bisector and opening angle

Calculate the bisector direction and the opening angle of the y-junctions per cell

#### Step 6a: ungrouped (new)

In [ ]:
%%time
projected = True
gdf_yjpnts = gpd.read_file("../Data/Crestlines/y_junction_points_updated_manual.shp")
gdf_yjpnts.set_index(gdf_yjpnts.columns[0], inplace=True)
gdf_yjpnts.rename_geometry("yj_points",inplace=True)
if projected:
    gdf_yjpnts.to_crs(3577,inplace=True)

gdf_yjlines = gpd.read_file("../Data/Crestlines/y_junction_lines_updated_manual.shp")
gdf_yjlines.set_index(gdf_yjlines.columns[0], inplace=True)
gdf_yjlines.rename_geometry("yj_lines",inplace=True)
if projected:
    gdf_yjlines.to_crs(3577,inplace=True)

gdf = pd.concat([gdf_yjpnts, gdf_yjlines],axis=1)
gdf.set_geometry("yj_lines",inplace=True)

# remove rings
gdf_exp = gdf.explode()
gdf = gdf[~gdf.index.isin(gdf_exp[gdf_exp.is_ring].index)]

radii = [100,250,500,750,1000,2000,5000] # rethink this!
out = gdf.apply(lambda x: pd.Series(get_bisectors_for_line_test(x.yj_points,gpd.GeoSeries(x.yj_lines.geoms),radii)),axis=1)
gdf_out = pd.concat([gdf,out],axis=1)
gdf_out.columns = [str(x) for x in gdf_out.columns]
gdf_out.columns = [col[:10] for col in gdf_out.columns]
gdf_out.set_geometry("yj_points", inplace =True)
gdf_out.drop("yj_lines",axis=1,inplace=True)
gdf_out.to_file("../Data/Crestlines/new_oang_bdir.shp")

#### Step 6b: grouped (new)

In [ ]:
%%time
### spatial join era 5 rect and the bisectors
gdf_rect = gpd.read_file("../Data/Crestlines/SKL_rect.shp")
gdf_out = gpd.read_file("../Data/Crestlines/new_oang_bdir.shp")
gdf_out.set_index("Index",inplace=True)
gdf_join = gpd.sjoin(gdf_rect, gdf_out.to_crs(4326))

### get the bisector groups
gdf_join_min = gdf_join[~np.isnan(gdf_join["100_oang"])]
gdf_join_min_group = gdf_join_min.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "minrad_oan":nanmean,"minrad_bdi":circnanmean})
gdf_join_min = pd.concat([gdf_join_min_group,gdf_join_min["index_poly"].value_counts()],axis=1)
gdf_join_min.rename(columns={"count":"count_min"},inplace=True)

gdf_join_100 = gdf_join[~np.isnan(gdf_join["100_oang"])]
gdf_join_100_group = gdf_join_100.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "100_oang":nanmean,"100_bdir":circnanmean})
gdf_join_100 = pd.concat([gdf_join_100_group,gdf_join_100["index_poly"].value_counts()],axis=1)
gdf_join_100.rename(columns={"count":"count_100"},inplace=True)

gdf_join_250 = gdf_join[~np.isnan(gdf_join["250_oang"])]
gdf_join_250_group = gdf_join_250.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "250_oang":nanmean,"250_bdir":circnanmean})
gdf_join_250 = pd.concat([gdf_join_250_group,gdf_join_250["index_poly"].value_counts()],axis=1)
gdf_join_250.rename(columns={"count":"count_250"},inplace=True)

gdf_join_500 = gdf_join[~np.isnan(gdf_join["500_oang"])]
gdf_join_500_group = gdf_join_500.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "500_oang":nanmean,"500_bdir":circnanmean})
gdf_join_500 = pd.concat([gdf_join_500_group,gdf_join_500["index_poly"].value_counts()],axis=1)
gdf_join_500.rename(columns={"count":"count_500"},inplace=True)

gdf_join_750 = gdf_join[~np.isnan(gdf_join["750_oang"])]
gdf_join_750_group = gdf_join_750.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "750_oang":nanmean,"750_bdir":circnanmean})
gdf_join_750 = pd.concat([gdf_join_750_group,gdf_join_750["index_poly"].value_counts()],axis=1)
gdf_join_750.rename(columns={"count":"count_750"},inplace=True)

gdf_join_1000 = gdf_join[~np.isnan(gdf_join["1000_oang"])]
gdf_join_1000_group = gdf_join_1000.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "1000_oang":nanmean,"1000_bdir":circnanmean})
gdf_join_1000 = pd.concat([gdf_join_1000_group,gdf_join_1000["index_poly"].value_counts()],axis=1)
gdf_join_1000.rename(columns={"count":"count_1000"},inplace=True)

gdf_join_2000 = gdf_join[~np.isnan(gdf_join["2000_oang"])]
gdf_join_2000_group = gdf_join_2000.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "2000_oang":nanmean,"2000_bdir":circnanmean})
gdf_join_2000 = pd.concat([gdf_join_2000_group,gdf_join_2000["index_poly"].value_counts()],axis=1)
gdf_join_2000.rename(columns={"count":"count_2000"},inplace=True)

gdf_join_5000 = gdf_join[~np.isnan(gdf_join["5000_oang"])]
gdf_join_5000_group = gdf_join_5000.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "5000_oang":nanmean,"5000_bdir":circnanmean})
gdf_join_5000 = pd.concat([gdf_join_5000_group,gdf_join_5000["index_poly"].value_counts()],axis=1)
gdf_join_5000.rename(columns={"count":"count_5000"},inplace=True)

gdf_join_max = gdf_join[~np.isnan(gdf_join["maxrad_oan"])]
gdf_join_max_group = gdf_join_max.groupby("index_poly").agg({"lat_idx":"first","lon_idx":"first","lat":"first","lon":"first",
                                               "maxrad_oan":nanmean,"maxrad_bdi":circnanmean})
gdf_join_max = pd.concat([gdf_join_max_group,gdf_join_max["index_poly"].value_counts()],axis=1)
gdf_join_max.rename(columns={"count":"count_max"},inplace=True)

# RERUN!
gdf_join_min.to_csv("../Data/Crestlines/oang_bdir_era5_min.csv", index=True)
gdf_join_100.to_csv("../Data/Crestlines/oang_bdir_era5_100.csv", index=True)
gdf_join_250.to_csv("../Data/Crestlines/oang_bdir_era5_250.csv", index=True)
gdf_join_500.to_csv("../Data/Crestlines/oang_bdir_era5_500.csv", index=True)
gdf_join_750.to_csv("../Data/Crestlines/oang_bdir_era5_750.csv", index=True)
gdf_join_1000.to_csv("../Data/Crestlines/oang_bdir_era5_1000.csv", index=True)
gdf_join_2000.to_csv("../Data/Crestlines/oang_bdir_era5_2000.csv", index=True)
gdf_join_5000.to_csv("../Data/Crestlines/oang_bdir_era5_5000.csv", index=True)
gdf_join_max.to_csv("../Data/Crestlines/oang_bdir_era5_max.csv", index=True)

### Step 7: Sign dune crests

Below we load the averaged unsingned angles per era5 cell and sign them by the 100 m bisector to identify the overall signed directions of crests and use that as a guide to spot weird looking cells on the fringes of dune fields and near lakes for example. Semi-automatic approach that requires loading in the lines and spot areas that need "flipping".

In [ ]:
# load average angle
gdf_ang = gpd.read_file("../Data/Crestlines/Average_angle_3577_from_sects_100.shp")
gdf_ang.set_index(gdf_ang.columns[0], inplace=True)

df_bis_100  = pd.read_csv("../Data/Crestlines/oang_bdir_era5_100.csv",index_col=0 ).iloc[:,-3:]

# load manual mask and data
gdfmaskflip = gpd.read_file("../Data/Crestlines/SKL_rect_flip_points.shp")
gdfmaskremove = gpd.read_file("../Data/Crestlines/SKL_rect_delete_points.shp") #unessary if used in the first step but shouldn't hurt 
gdf_rect = gpd.read_file("../Data/Crestlines/SKL_rect.shp")
gdf = pd.concat([gdf_ang.loc[:,["lon_idx","lat_idx","lat", "lon","lon_3577", "lat_3577", "linear_ang","linear_std","geometry"]],df_bis_100] ,axis=1)

# sign crests by 100m bisectors
gdf["avg100signed"] = sign_crests(gdf.linear_ang,range_neg_pi_to_pi(gdf["100_bdir"]))

# remove the 
idx_to_flip = gpd.sjoin(gdf_rect,gdfmaskflip).index_poly.unique()
idx_to_remove = gpd.sjoin(gdf_rect,gdfmaskremove).index_poly.unique()
gdf.loc[idx_to_flip,"avg100signed"] = range_neg_pi_to_pi(gdf.loc[idx_to_flip,"avg100signed"] + np.pi)
gdf = gdf[~gdf.index.isin(idx_to_remove)]
gdf_rect_masked = gdf_rect[~gdf_rect.index_poly.isin(idx_to_remove)].set_index("index_poly")

# export 
lonlatbounds = [[112.9,153.4],[-38.9,-10.5]]
res  = 0.1

lon = np.round(np.arange(lonlatbounds[0][0],lonlatbounds[0][1]+1e-4,res),1)
lat = np.round(np.arange(lonlatbounds[1][0],lonlatbounds[1][1]+1e-4,res),1)

# save the updated signed raster and the era5 cells 
gdf_var_to_raster(gdf,"avg100signed", "../Data/Crestlines/signed_and_manual.tif",np.nan,lon,lat,crs=4326)
gdf_rect_masked.to_file("../Data/Crestlines/SKL_rect_masked.shp", index=True)

#update orientation in linestrings and save
gdf.loc[:,"geometry"] = gdf.apply(lambda x: generate_linestring(x.lon_3577,x.lat_3577,x.avg100signed,180*30),axis=1).values
gdf.rename(columns={"avg100signed":"avg100sig"}).to_file("../Data/Crestlines/signed_average_angle.shp", index=True)

### Step 8: Slopes and height

In [ ]:
save=False

SKL_file = "../Data/Crestlines/Lines_per_cell_3577_from_sects_100.shp"
SKL_rect = "../Data/Crestlines/SKL_rect_masked.shp"
SKL_ang = "../Data/Crestlines/signed_average_angle.shp"
COP_files = sorted(list(glob.glob("../Data/Copernicus_OpenTopo_Australia/*.tif")))

COP_dict = {}
for i,file in enumerate(COP_files):
    COP_dict[i] = file

gdf_ang = gpd.read_file(SKL_ang)
gdf_ang = gdf_ang.set_index("index_poly")

gdf_rect = gpd.read_file(SKL_rect)
gdf_rect = gdf_rect.set_index("index_poly")
gdf = gpd.read_file(SKL_file) # load all lines 
gdf = gdf.set_index("index_poly")
gdf = pd.concat([gdf,gdf_rect.rename_geometry("geom_poly")],join="inner",axis=1)

gdf["clipped_lines"] = gdf.geometry.to_crs(4326).intersection(gdf.geom_poly,align=False)
gdf["clipped_lines_3577"] = gdf.clipped_lines.to_crs(3577)
gdf["geometry_3577"] = gdf.geometry.to_crs(3577)
gdf["COPs"] = gdf.apply(lambda x: get_cop_files(x.lon, x.lat, COP_dict),axis=1)
gdf = pd.concat([gdf,gdf_ang.loc[:,["linear_ang","linear_std","avg100sig"]]],join="inner",axis=1)
gdf["geometry"] = gdf.geometry.to_crs(4326)

out = gdf.apply(lambda x: handle_rasterio_new(x,COP_dict,samp_inter=100,samp_offset=30),axis=1)
dft = pd.DataFrame(out.tolist(), index=out.index, columns=['left_ang_deg_mean', 'right_ang_deg_mean',"mean_slope_mean", "left_minus_right_ang_mean", 'rel_height_mean', 'width_mean',
                                                           "rel_height_shallow_mean","width_shallow_mean","max_height_mean",
                                                           'left_ang_deg_median', 'right_ang_deg_median',"mean_slope_median", "left_minus_right_ang_median", 'rel_height_median', 'width_median',
                                                           "rel_height_shallow_median","width_shallow_median","max_height_median",
                                                           'volume_sum','volume_mean','volume_median','volume_shallow_sum','volume_shallow_mean','volume_shallow_median'])
# if save:
#     dft.to_csv("../Data/Crestlines/vol_slope_height_width_si100_so30.csv",index=True)